> **Synthetic data.** Split identifiers loaded below come from `data/synthetic/SYNTHETIC_tvt_info_split_*.pkl` (fake ids starting at 920000). The lab/vital table is the seeded mock, not MIMIC-III. See `DATA.md`.


In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import softmax

import json
import os

In [ ]:
from pathlib import Path

def _repo_file(rel):
    """Resolve a repo-relative path from either the repo root or notebooks/."""
    for root in (Path.cwd(), Path.cwd().parent):
        candidate = root / rel
        if candidate.exists():
            return str(candidate)
    return str((Path.cwd().parent / rel))

Path('images').mkdir(parents=True, exist_ok=True)
Path('images/eda').mkdir(parents=True, exist_ok=True)


In [ ]:
# from preprocessing_mio import load_data, get_dataset

In [ ]:
matrix = [[1, -0.25, -0.18, -0.2],
[-0.25, 1, -0.17, -0.018],
[-0.18, -0.17, 1, -0.0029],
[-0.2, -0.018, -0.0029, 1]]
matrix_df = pd.DataFrame(matrix, columns=['a', 'b', 'c', 'd'], index=['1','2','3','4'])

In [ ]:
labels = ['score_weight', 'score_cover', 'score_gain', 'nulls']

fig, ax = plt.subplots(figsize=(15,12))
sns.heatmap(matrix, annot=True, ax=ax, vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels)
plt.yticks(rotation=0) 
plt.savefig('images/corr_xgboost.png', bbox_inches='tight')
plt.show()

---------------------

# Mejores hiperparametros por modelo

In [ ]:
best_hyperparameters_rnn = pd.read_csv(_repo_file('output/rnn/best_hyperparameters.csv'))
best_hyperparameters_rnn.sort_values(['model_name', 'split'])


In [ ]:
rnn_scores_val_auprc = pd.read_csv(_repo_file('output/rnn/scores_val_auprc.csv'))
auprc_scores_val_auprc = rnn_scores_val_auprc[rnn_scores_val_auprc['score_type']=='auprc']
auprc_scores_val_auprc.columns = ['model_name'] + list(auprc_scores_val_auprc.columns.values[1:])


In [ ]:
best_hyperparameters_rnn.merge(auprc_scores_val_auprc, on=['model_name', 'split'])

In [ ]:
fig, ax = plt.subplots(figsize=(15,12))
sns.heatmap(best_hyperparameters_rnn.merge(auprc_scores_val_auprc, on=['model_name', 'split']).drop(columns=['horizon', 'split']).corr(numeric_only=True), annot=True, ax=ax, vmin=-1, vmax=1)
plt.yticks(rotation=0) 
plt.savefig('images/corr_hiperparams_rnn.png', bbox_inches='tight')
plt.show()

In [ ]:
best_hyperparameters_rnn.describe()

In [ ]:
best_hyperparameters_rnn['learning_rate'].quantile([0.89])

------------------------------

In [ ]:
def plot_learning_curves(early_stopping_monitor, savepath=None, show=True):
    splits = [0,1,2]

    files_folder_path = f'./images/train/{early_stopping_monitor}'

    rows_limit = 10
    figsize = (20,40)

    i = 0
    total = 0
    suptitles = ['gru_stacked', 'gru_denses_stacked_dropout', 'lstm_denses_stacked_dropout', 'baseline_lstm', 'gru_stacked_denses_stacked_dropout', 'lstm_stacked_denses_stacked_dropout', 'gru_denses_stacked', 'lstm_stacked', 'baseline_gru', 'lstm_denses_stacked']

    fig, axs = plt.subplots(rows_limit, 3, figsize=figsize, constrained_layout=False)
    # subfigs = fig.subfigures(nrows=rows_limit, ncols=1, hspace=4)
    # for row, subfig in enumerate(subfigs):
    #     subfig.suptitle(suptitles[row])

    # left = 0.1
    # bottom = 0.1
    # right = 0.9
    # top = 1
    # wspace = 0.4
    # hspace = 1

    # plt.subplots_adjust(left=left,
    #                     bottom=bottom, 
    #                     right=right, 
    #                     top=top,
    #                     wspace=wspace, 
    #                     hspace=hspace)

    if not os.path.isdir(files_folder_path):
        rng = np.random.default_rng(0)
        steps = np.arange(1, 21)
        for folder in suptitles:
            folder_path = os.path.join(files_folder_path, folder)
            os.makedirs(folder_path, exist_ok=True)
            for split in splits:
                train = pd.DataFrame({
                    "Step": steps,
                    "Value": 1.0 / steps + rng.normal(0, 0.01, size=len(steps)),
                })
                validation = pd.DataFrame({
                    "Step": steps,
                    "Value": 1.2 / steps + rng.normal(0, 0.02, size=len(steps)),
                })
                train.to_csv(os.path.join(folder_path, f"train_loss_split_{split}.csv"), index=False)
                validation.to_csv(os.path.join(folder_path, f"validation_loss_split_{split}.csv"), index=False)

    for folder in os.listdir(files_folder_path):

        if i==rows_limit:
            plt.show()

            fig, axs = plt.subplots(rows_limit, 3, figsize=figsize, constrained_layout=False)
            # subfigs = fig.subfigures(nrows=rows_limit, ncols=1, hspace=.75)
            # for row, subfig in enumerate(subfigs):
            #     subfig.suptitle(f'{folder}')

            # plt.subplots_adjust(left=left,
            #             bottom=bottom, 
            #             right=right, 
            #             top=top,
            #             wspace=wspace, 
            #             hspace=hspace)

            i=0
        
        for split in splits:
            
            train_loss_path = f'train_loss_split_{split}.csv'
            train_loss = pd.read_csv(f'{files_folder_path}/{folder}/{train_loss_path}')

            validation_loss_path = f'validation_loss_split_{split}.csv'
            validation_loss = pd.read_csv(f'{files_folder_path}/{folder}/{validation_loss_path}')

            if folder == 'gru_denses_stacked_dropout' and split==1:
                axs[i, split].plot(train_loss['Step'], train_loss['Value'], label='Validación', color='tab:orange')
                axs[i, split].plot(validation_loss['Step'], validation_loss['Value'], label='Entrenamiento', color='tab:blue')
            elif folder == 'lstm_denses_stacked_dropout' and split==2:
                axs[i, split].plot(train_loss['Step'], train_loss['Value'], label='Validación', color='tab:orange')
                axs[i, split].plot(validation_loss['Step'], validation_loss['Value'], label='Entrenamiento', color='tab:blue')
            elif folder == 'gru_stacked_denses_stacked_dropout' and (split==0 or split==1):
                axs[i, split].plot(train_loss['Step'], train_loss['Value'], label='Validación', color='tab:orange')
                axs[i, split].plot(validation_loss['Step'], validation_loss['Value'], label='Entrenamiento', color='tab:blue')
            else:
                axs[i, split].plot(train_loss['Step'], train_loss['Value'], label='Entrenamiento', color='tab:blue')
                axs[i, split].plot(validation_loss['Step'], validation_loss['Value'], label='Validación', color='tab:orange')

            if split==1:
                axs[i, split].set_title(f'{folder}')
            
            axs[i,split].set_xlabel('Step')
            axs[i, split].set_ylabel('Loss')
            fig.subplots_adjust(hspace=0.35)
        
        total += 1
        i+=1

    fig.legend(['Entrenamiento', 'Validación'], loc='lower center', bbox_to_anchor=(0.5125, 0.1), ncol=2)
    
    if savepath:
        plt.savefig(savepath, bbox_inches='tight')
    if show:
        plt.show()


In [ ]:
plot_learning_curves('val_auprc', savepath='images/loss_training_rnn_val_auprc.png')

In [ ]:
plot_learning_curves('val_loss', savepath='images/loss_training_rnn_val_logloss.png', show=False)

--------------------

In [ ]:
hyperparam_opt_times = pd.read_csv(_repo_file('output/rnn/hyperparam_opt_times.csv'))
hyperparam_opt_times


In [ ]:
# Get time in minutes (it is in seconds)
hyperparam_opt_times['time'] = hyperparam_opt_times['time'] / 60
# hyperparam_opt_times.groupby('model').sum()
hyperparam_opt_times.sort_values(by=['model', 'split'])

In [ ]:
# Plot time for each model and split
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='time', y='model', data=hyperparam_opt_times)

plt.title('Tiempo para la optimización de hiperparámetros por red.')
plt.xlabel('Tiempo (min)')
plt.ylabel('Modelo')

# Change y axis labels names
ax.set_yticklabels(['baseline_lstm', 'baseline_gru', 'lstm_denses_stacked', 'gru_denses_stacked', 'lstm_denses_stacked_dropout', 'gru_denses_stacked_dropout', 'lstm_stacked', 'gru_stacked', 'lstm_stacked_denses_stacked_dropout', 'gru_stacked_denses_stacked_dropout'])

# handles, labels = ax.get_legend_handles_labels()
# labels = ['baseline_lstm', 'baseline_gru', 'lstm_denses_stacked', 'gru_denses_stacked', 'lstm_denses_stacked_dropout', 'gru_denses_stacked_dropout', 'lstm_stacked', 'gru_stacked', 'lstm_stacked_denses_stacked_dropout', 'gru_stacked_denses_stacked_dropout']
# plt.legend(handles, labels, bbox_to_anchor=(1, 0.75), ncol=1, title='Modelo')

# plt.savefig('images/hyperparam_opt_times_per_model.png', bbox_inches='tight')
plt.show()

In [ ]:
# Plot time for each model and split
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='time', y='split', hue='model', data=hyperparam_opt_times, orient='h')

plt.title('Tiempo para la optimización de hiperparámetros por red y split')
plt.xlabel('Tiempo (min)')
plt.ylabel('Split')

handles, labels = ax.get_legend_handles_labels()
labels = ['baseline_lstm', 'baseline_gru', 'lstm_denses_stacked', 'gru_denses_stacked', 'lstm_denses_stacked_dropout', 'gru_denses_stacked_dropout', 'lstm_stacked', 'gru_stacked', 'lstm_stacked_denses_stacked_dropout', 'gru_stacked_denses_stacked_dropout']
plt.legend(handles, labels, bbox_to_anchor=(1, 0.75), ncol=1, title='Modelo')

# plt.savefig('images/hyperparam_opt_times.png', bbox_inches='tight')
plt.show()

--------------

In [ ]:
rnn_scores_val_auprc = pd.read_csv(_repo_file('output/rnn/scores_val_auprc.csv'))
auprc_scores_val_auprc = rnn_scores_val_auprc[rnn_scores_val_auprc['score_type']=='auprc']
aucroc_scores_val_auprc = rnn_scores_val_auprc[rnn_scores_val_auprc['score_type']=='aucroc']
rnn_scores_val_auprc


In [ ]:
rnn_scores_val_loss = pd.read_csv(_repo_file('output/rnn/scores_val_loss.csv'))
auprc_scores_val_loss = rnn_scores_val_loss[rnn_scores_val_loss['score_type']=='auprc']
aucroc_scores_val_loss = rnn_scores_val_loss[rnn_scores_val_loss['score_type']=='aucroc']
rnn_scores_val_loss


# Scores

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10,15))
sns.lineplot(x='horizon', y='score', hue='model_type', data=auprc_scores_val_auprc, errorbar=None, ax=axs[0])
axs[0].invert_xaxis()
axs[0].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[0].set_ylabel('AUPRC')
axs[0].get_legend().remove()

sns.barplot(x='horizon', y='score', hue='model_type', data=auprc_scores_val_auprc, errorbar='sd', ax=axs[1])
axs[1].invert_xaxis()
axs[1].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[1].set_ylabel('AUPRC')
axs[1].get_legend().remove()

handles, labels = axs[0].get_legend_handles_labels()
# labels = ['Modelo base-line', 'Modelo AUPRC', 'Modelo AUPRC media', 'Modelo AUPRC  decr.', 'Modelo AUPRC crec.']
labels = ['baseline_lstm', 'baseline_gru', 'lstm_denses_stacked', 'gru_denses_stacked', 'lstm_denses_stacked_dropout', 'gru_denses_stacked_dropout', 'lstm_stacked', 'gru_stacked', 'lstm_stacked_denses_stacked_dropout', 'gru_stacked_denses_stacked_dropout']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.07), ncol=2)

plt.savefig('images/lineplot_barplot_auprc_rnn.png', bbox_inches='tight')
plt.show()


In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10,15))
sns.barplot(x='horizon', y='score', hue='model_type', data=auprc_scores_val_auprc, errorbar='sd', ax=axs[0])
axs[0].invert_xaxis()
axs[0].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[0].set_ylabel('AUPRC')
axs[0].set_title('AUPRC\nEntrenamiento con early stopping sobre AUPRC de validación')
axs[0].get_legend().remove()

sns.barplot(x='horizon', y='score', hue='model_type', data=auprc_scores_val_loss, errorbar='sd', ax=axs[1])
axs[1].invert_xaxis()
axs[1].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[1].set_ylabel('AUPRC')
axs[1].set_title('AUPRC\nEntrenamiento con early stopping sobre loss de validación')
axs[1].get_legend().remove()

handles, labels = axs[0].get_legend_handles_labels()
# labels = ['Modelo base-line', 'Modelo AUPRC', 'Modelo AUPRC media', 'Modelo AUPRC  decr.', 'Modelo AUPRC crec.']
labels = ['baseline_lstm', 'baseline_gru', 'lstm_denses_stacked', 'gru_denses_stacked', 'lstm_denses_stacked_dropout', 'gru_denses_stacked_dropout', 'lstm_stacked', 'gru_stacked', 'lstm_stacked_denses_stacked_dropout', 'gru_stacked_denses_stacked_dropout']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.07), ncol=2)

# plt.savefig('images/lineplot_barplot_auprc_rnn.png', bbox_inches='tight')
plt.show()


# Times

In [ ]:
rnn_scores_val_auprc['time'] = rnn_scores_val_auprc['time'] / 60
rnn_scores_val_loss['time'] = rnn_scores_val_loss['time'] / 60

In [ ]:
rnn_scores_val_auprc.groupby('model_type').head(1)

In [ ]:
rnn_scores_val_loss.groupby('model_type').head(1)

In [ ]:
# 2x1 subplots
fig, ax = plt.subplots(1, 1, figsize=(10,8))

sns.barplot(x='time', y='model_type', data=rnn_scores_val_auprc.groupby('model_type').head(1))
ax.set_xlabel('Tiempo de entrenamiento y evaluación (minutos)')
ax.set_ylabel('Modelo')
plt.savefig('images/train_test_time_rnn.png', bbox_inches='tight')
plt.show()

-------------------

In [ ]:
horizons = list(range(7, -1, -1))

decreasing_weights = softmax([horizon for horizon in horizons])
increasing_weights = softmax([horizon for horizon in horizons][::-1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=[7,6,5,4,3,2,1,0], y=decreasing_weights, ax=axes[0])
axes[0].invert_xaxis()
axes[0].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axes[0].set_ylabel('Peso')

sns.barplot(x=list(range(7, -1, -1)), y=increasing_weights, ax=axes[1])
axes[1].invert_xaxis()
axes[1].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axes[1].set_ylabel('Peso')

axes[0].set_title('AUPRC decreciente - softmax([7,..,0])')
axes[1].set_title('AUPRC creciente - softmax([0,..,7])')

# plt.savefig('images/auprc_weights.png', bbox_inches='tight')
plt.show()

---------------------------

In [ ]:
import pickle

# Load pickle file
def load_pickle(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [ ]:
tvt_info_split_0 = load_pickle(_repo_file('data/synthetic/SYNTHETIC_tvt_info_split_0.pkl'))
tvt_info_split_1 = load_pickle(_repo_file('data/synthetic/SYNTHETIC_tvt_info_split_1.pkl'))
tvt_info_split_2 = load_pickle(_repo_file('data/synthetic/SYNTHETIC_tvt_info_split_2.pkl'))


In [ ]:
len(tvt_info_split_0['test']), len(tvt_info_split_1['test']), len(tvt_info_split_2['test'])

In [ ]:
len(np.intersect1d(tvt_info_split_0['test'], tvt_info_split_1['test'])), len(np.intersect1d(tvt_info_split_0['test'], tvt_info_split_2['test'])), len(np.intersect1d(tvt_info_split_1['test'], tvt_info_split_2['test']))

In [ ]:
63*100/620

---------------------

In [ ]:
clockify_times = pd.DataFrame({
    "Description": [f"SYN-task-{i}" for i in range(1, 8)],
    "Time (h)": [12, 38, 75, 88, 27, 33, 18],
    "Amount (USD)": [0, 0, 0, 0, 0, 0, 0],
    "tarea": [f"SYN-task-{i}" for i in range(1, 8)],
    "tiempo real": [12, 38, 75, 88, 27, 33, 18],
})
clockify_times


In [ ]:
clockify_times = clockify_times.drop(columns=['Description', 'Time (h)', 'Amount (USD)'])
clockify_times.columns = ['tarea', 'tiempo real']
clockify_times['tiempo previsto'] = [10, 40, 80, 90, 30, 35, 15]

In [ ]:
clockify_times_sns_format = []
for index, row in clockify_times.iterrows():
    clockify_times_sns_format.append([row['tarea'], row['tiempo previsto'], 'Planificado'])
    clockify_times_sns_format.append([row['tarea'], row['tiempo real'], 'Real'])

clockify_times_sns_format = pd.DataFrame(clockify_times_sns_format, columns=['Tarea', 'Tiempo total', 'Tiempo'])
clockify_times_sns_format

In [ ]:
sns.barplot(x='Tiempo total', y='Tarea', hue='Tiempo', data=clockify_times_sns_format, orient='h')
# plt.savefig('images/desviacion_temporal_tareas_clockify.png', bbox_inches='tight')
plt.show()

In [ ]:
difference = clockify_times.copy()
difference['Desviación'] = difference['tiempo real'] - difference['tiempo previsto'] 
difference.columns = ['Tarea', 'Tiempo real', 'Tiempo previsto', 'Desviación']
difference

In [ ]:
fig, ax = plt.subplots(ncols=1, figsize=(10, 5))

sns.barplot(data=difference, x='Desviación', y='Tarea', 
            orient='h', ax=ax)
ax.set_xlabel('Desviación temporal (horas)')

# plt.savefig('images/desviacion_temporal_tareas_clockify.png', bbox_inches='tight')
plt.show()


-----------------------------

In [ ]:
splits = [0,1,2]
fig, axs = plt.subplots(3, 2, figsize=(10,10))
plt.setp(axs, ylim=(0.45, 1))

for split in splits:
    with open(_repo_file(f'output/xgboost/evals_split_{split}_model_baseline.json')) as json_file:
        baseline_evals = json.load(json_file)
    with open(_repo_file(f'output/xgboost/evals_split_{split}_model_tuned.json')) as json_file:
        tuned_evals = json.load(json_file)

    axs[split, 0].plot(range(len(baseline_evals['validation_0']['auprc'])), list(map(lambda x: -x, baseline_evals['validation_0']['auprc'])), label='Entrenamiento')
    axs[split, 0].plot(range(len(baseline_evals['validation_1']['auprc'])), list(map(lambda x: -x, baseline_evals['validation_1']['auprc'])), label='Validación')
    axs[split, 0].set_xlabel('Epoch')
    axs[split, 0].set_ylabel('AUPRC')

    axs[split, 1].plot(range(len(tuned_evals['validation_0']['auprc'])), list(map(lambda x: -x, tuned_evals['validation_0']['auprc'])), label='Entrenamiento')
    axs[split, 1].plot(range(len(tuned_evals['validation_0']['auprc'])), list(map(lambda x: -x, tuned_evals['validation_1']['auprc'])), label='Validación')
    axs[split, 1].set_xlabel('Epoch')
    axs[split, 1].set_ylabel('AUPRC')

    axs[split, 0].set_title(f'Split {split}')
    axs[split, 1].set_title(f'Split {split}')

    handles, labels = axs[split, 1].get_legend_handles_labels()

fig.suptitle('       Modelo base-line                                    Modelo optimizado', fontsize=15, y=0.95)
fig.subplots_adjust(hspace=.5)

fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.07), ncol=2)
# plt.savefig('images/auprc_training_xgboost.png', bbox_inches='tight')
plt.show()


In [ ]:
splits = [0,1,2]
fig, axs = plt.subplots(3, 2, figsize=(10,10))
# plt.setp(axs, ylim=(0.45, 1))

for split in splits:
    with open(_repo_file(f'output/xgboost/evals_split_{split}_model_baseline.json')) as json_file:
        baseline_evals = json.load(json_file)
    with open(_repo_file(f'output/xgboost/evals_split_{split}_model_tuned.json')) as json_file:
        tuned_evals = json.load(json_file)

    axs[split, 0].plot(range(len(baseline_evals['validation_0']['logloss'])), list(map(lambda x: x, baseline_evals['validation_0']['logloss'])), label='Entrenamiento')
    axs[split, 0].plot(range(len(baseline_evals['validation_1']['logloss'])), list(map(lambda x: x, baseline_evals['validation_1']['logloss'])), label='Validación')
    axs[split, 0].set_xlabel('Epoch')
    axs[split, 0].set_ylabel('Log loss')

    axs[split, 1].plot(range(len(tuned_evals['validation_0']['logloss'])), list(map(lambda x: x, tuned_evals['validation_0']['logloss'])), label='Entrenamiento')
    axs[split, 1].plot(range(len(tuned_evals['validation_0']['logloss'])), list(map(lambda x: x, tuned_evals['validation_1']['logloss'])), label='Validación')
    axs[split, 1].set_xlabel('Epoch')
    axs[split, 1].set_ylabel('Log loss')

    axs[split, 0].set_title(f'Split {split}')
    axs[split, 1].set_title(f'Split {split}')

    handles, labels = axs[split, 1].get_legend_handles_labels()

fig.suptitle('       Modelo base-line                                    Modelo optimizado', fontsize=15, y=0.95)
fig.subplots_adjust(hspace=.5)

fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.07), ncol=2)
plt.savefig('images/logloss_training_xgboost.png', bbox_inches='tight')
plt.show()


-------------------------

In [ ]:
vitals_features = ['sysbp', 'diabp', 'meanbp', 'resprate', 'heartrate',
    'spo2_pulsoxy', 'tempc', 'cardiacoutput', 'tvset', 'tvobserved',
    'tvspontaneous', 'peakinsppressure', 'totalpeeplevel', 'o2flow',
    'fio2']
labs_features = ['albumin', 'bands', 'bicarbonate', 'bilirubin',
    'creatinine', 'chloride', 'glucose', 'hematocrit', 'hemoglobin',
    'lactate', 'platelet', 'potassium', 'ptt', 'inr', 'pt', 'sodium',
    'bun', 'wbc', 'creatinekinase', 'ck_mb', 'fibrinogen', 'ldh',
    'magnesium', 'calcium_free', 'po2_bloodgas', 'ph_bloodgas',
    'pco2_bloodgas', 'so2_bloodgas', 'troponin_t']
all_features = vitals_features + labs_features

In [ ]:
feature_importances = pd.read_csv(_repo_file('output/xgboost/feature_importances.csv'))
full_labvitals = pd.read_csv(_repo_file('data/synthetic/SYNTHETIC_full_labvitals_horizon_0_dropped_short.csv'))


In [ ]:
nulls = full_labvitals[all_features].isnull().sum()
# Plot nulls in ascending order
nulls.sort_values(ascending=True, inplace=True)
nulls.plot(kind='barh', figsize=(12, 10))
plt.title('Nulos por característica del modelo.')
# plt.savefig('images/eda/nulls_lab_vital.png', bbox_inches='tight')
plt.show()

In [ ]:
feature_importances['nulls'] = feature_importances['feature'].map(nulls)

In [ ]:
# Create 3 empty columns for score_weight, score_gain and score_cover
feature_importances['score_weight'] = 0.0
feature_importances['score_gain'] = 0.0
feature_importances['score_cover'] = 0.0

# For each row set a score in these columns based on the importance_type
for index, row in feature_importances.iterrows():
    if row['importance_type'] == 'weight':
        feature_importances.loc[index, 'score_weight'] = row['score']
    elif row['importance_type'] == 'gain':
        feature_importances.loc[index, 'score_gain'] = row['score']
    elif row['importance_type'] == 'cover':
        feature_importances.loc[index, 'score_cover'] = row['score']

feature_importances = feature_importances.drop(columns=['score'])

In [ ]:
feature_importances['feature'] = feature_importances['feature'].astype('category').cat.codes
feature_importances['importance_type'] = feature_importances['importance_type'].astype('category').cat.codes
feature_importances

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(feature_importances[['score_weight', 'score_cover', 'score_gain', 'nulls']].corr(numeric_only=True), annot=True, ax=ax)
plt.yticks(rotation=0)
# plt.savefig('images/corr_xgboost.png', bbox_inches='tight')
plt.show()

---------------------------------------------

In [ ]:
xgboost_scores = pd.read_csv(_repo_file('output/xgboost/scores.csv'))
xgboost_scores


In [ ]:
auprc_scores = xgboost_scores[xgboost_scores['score_type']=='auprc']
aucroc_scores = xgboost_scores[xgboost_scores['score_type']=='aucroc']

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10,15))
sns.lineplot(x='horizon', y='score', hue='model_type', data=auprc_scores, errorbar='sd', n_boot=10000, ax=axs[0])
axs[0].invert_xaxis()
axs[0].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[0].set_ylabel('AUPRC')
axs[0].get_legend().remove()

# sns.lineplot(x='horizon', y='score', hue='model_type', data=aucroc_scores, errorbar='sd', n_boot=10000, ax=axs[1])
sns.barplot(x='horizon', y='score', hue='model_type', data=auprc_scores, errorbar='sd', ax=axs[1])
axs[1].invert_xaxis()
axs[1].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[1].set_ylabel('AUC-ROC')
axs[1].get_legend().remove()

handles, labels = axs[0].get_legend_handles_labels()
labels = ['Modelo base-line', 'Modelo AUPRC', 'Modelo AUPRC media', 'Modelo AUPRC  decr.', 'Modelo AUPRC crec.']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.07), ncol=2)

# plt.savefig('images/lineplot_barplot_auprc_xgboost.png', bbox_inches='tight')
plt.show()


--------------

In [ ]:
rnn_scores = pd.concat([
    pd.read_csv(_repo_file("output/rnn/scores_val_auprc.csv")),
    pd.read_csv(_repo_file("output/rnn/scores_val_loss.csv")),
], ignore_index=True)
rnn_scores


In [ ]:
auprc_scores = rnn_scores[rnn_scores['score_type']=='auprc']
aucroc_scores = rnn_scores[rnn_scores['score_type']=='aucroc']

In [ ]:
auprc_scores = auprc_scores[auprc_scores['model_type'].str.contains('(val_loss)', regex=False) == False]
aucroc_scores = aucroc_scores[aucroc_scores['model_type'].str.contains('(val_loss)', regex=False) == False]

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10,15))
sns.lineplot(x='horizon', y='score', hue='model_type', data=auprc_scores, errorbar=None, n_boot=10000, ax=axs[0])
axs[0].invert_xaxis()
axs[0].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[0].set_ylabel('AUPRC')
axs[0].get_legend().remove()

# sns.lineplot(x='horizon', y='score', hue='model_type', data=aucroc_scores, errorbar='sd', n_boot=10000, ax=axs[1])
sns.barplot(x='horizon', y='score', hue='model_type', data=auprc_scores, errorbar='sd', ax=axs[1])
axs[1].invert_xaxis()
axs[1].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[1].set_ylabel('AUPRC')
axs[1].get_legend().remove()

handles, labels = axs[0].get_legend_handles_labels()
# labels = ['Modelo base-line', 'Modelo AUPRC', 'Modelo AUPRC media', 'Modelo AUPRC  decr.', 'Modelo AUPRC crec.']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.07), ncol=2)

# plt.savefig('images/lineplot_barplot_auprc_xgboost.png', bbox_inches='tight')
plt.show()


In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10,15))
sns.lineplot(x='horizon', y='score', hue='model_type', data=aucroc_scores, errorbar='sd', n_boot=10000, ax=axs[0])
axs[0].invert_xaxis()
axs[0].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[0].set_ylabel('AUPRC')
axs[0].get_legend().remove()

# sns.lineplot(x='horizon', y='score', hue='model_type', data=aucroc_scores, errorbar='sd', n_boot=10000, ax=axs[1])
sns.barplot(x='horizon', y='score', hue='model_type', data=aucroc_scores, errorbar='sd', ax=axs[1])
axs[1].invert_xaxis()
axs[1].set_xlabel('Horas del horizonte (previas al momento de aparición)')
axs[1].set_ylabel('AUPRC')
axs[1].get_legend().remove()

handles, labels = axs[0].get_legend_handles_labels()
# labels = ['Modelo base-line', 'Modelo AUPRC', 'Modelo AUPRC media', 'Modelo AUPRC  decr.', 'Modelo AUPRC crec.']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.07), ncol=2)

# plt.savefig('images/lineplot_barplot_auprc_xgboost.png', bbox_inches='tight')
plt.show()


In [ ]:
# fig, axs = plt.subplots(2, figsize=(10,15))
# sns.lineplot(x='horizon', y='score', hue='model_type', data=xgboost_scores, errorbar='sd', n_boot=10000, ax=axs[0])
# axs[0].invert_xaxis()
# axs[0].set_xlabel('Horas del horizonte (previas al momento de aparición)')
# axs[0].set_ylabel('AUPRC')
# axs[0].legend(loc='upper left')

# auprc_scores_averaged_thres = xgboost_scores.copy()
# auprc_scores_averaged_thres['score'] -= 0.4

# sns.barplot(x='horizon', y='score', hue='model_type', data=auprc_scores_averaged_thres, errorbar='sd', ax=axs[1])
# axs[1].invert_xaxis()
# axs[1].set_xlabel('Horas del horizonte (previas al momento de aparición)')
# axs[1].set_ylabel('AUPRC')
# axs[1].legend(loc='upper left')

# # plt.savefig('images/lineplot_barplot_auprc_xgboost.png', bbox_inches='tight')
# plt.show()
